In [1]:
import json
from pathlib import Path
from typing import Any
import itertools as it
import wandb
from wandb.apis.public.runs import Run, Runs

from utility import LOCAL_DATA_PATH

In [2]:
api = wandb.Api()

In [3]:
runs = api.runs(
    "graphcore/llama-mobile",
    filters={"config.name": "sweep-05-02-26", "state": "finished"},
)
baseline_runs = api.runs(
    "graphcore/llama-mobile",
    filters={"config.name": "baseline-12-02-26", "state": "finished"},
)
all_runs = list(it.chain(baseline_runs, runs))

In [4]:
def get_run(runs: Runs, name: str) -> Run | None:
    for run in runs:
        if run.name == name:
            return run


def get_task_outputs(run: Run) -> dict[str, Any]:
    artifacts = [a for a in run.logged_artifacts() if a.type == "task_outputs"]
    assert len(artifacts) == 1, "Found none or multiple matches"
    a = artifacts[0]
    a_dir = a.download(
        Path(LOCAL_DATA_PATH).parent / f"artifacts/{run.name}/task_outputs"
    )
    out = {}
    for path in Path(a_dir).rglob("*.jsonl"):
        with path.open() as f:
            out[path.stem] = [json.loads(line) for line in f]
    return out

In [5]:
# Runs:
# - ethereal-shape-1606: quantised, no training
# - whole-fire-1607: bfloat16
# - wandering-breeze-1570: lr=2**-16, n_steps=2048
run_b = get_run(all_runs, "whole-fire-1607")
run_0 = get_run(all_runs, "ethereal-shape-1606")
run_2k = get_run(all_runs, "wandering-breeze-1570")
out_b = get_task_outputs(run_b)
out_0 = get_task_outputs(run_0)
out_2k = get_task_outputs(run_2k)

wandb:   4 of 4 files downloaded.  
wandb:   4 of 4 files downloaded.  
wandb:   4 of 4 files downloaded.  


In [6]:
from eval import vqa

for k in out_2k:
    task = vqa.TASKS[k]
    for x, y in zip(out_2k[k], out_b[k]):
        x.update(
            **task.evaluate_prediction(
                x["output"], y["answers"], include_relaxed_metrics=True
            ), answers=y["answers"]
        )

In [7]:
fail_examples = [x for x in out_2k["docvqa"] if x["accuracy_relaxed"] == 0.0]

In [8]:
for e in fail_examples[0:25]:
    print("OUTPUT: ", e["output"])
    print("CORRECT: ", e["answers"])
    print()

OUTPUT:  **Cellular Responsiveness and Physiological Adaptation**

The research budget increment for this department is the lowest in FY 74, with a total increment of 2,300.<|eot_id|>
CORRECT:  ['Immunology']

OUTPUT:  The 'catalog sales' department comes under the SE Vacations division of the company.<|eot_id|>
CORRECT:  ['se products', 'SE Products']

OUTPUT:  The Interobserver for BG1 is 20.<|eot_id|>
CORRECT:  ['.97']

OUTPUT:  The total number of options granted in 2006 is 1,08,35,696.<|eot_id|>
CORRECT:  ['60,95,625']

OUTPUT:  The lowest year-on-year growth is in December 2008.<|eot_id|>
CORRECT:  ['Dec-08']

OUTPUT:  The purchase order number is 99-19/6866.<|eot_id|>
CORRECT:  ['99-19686', '99 - 19686']

OUTPUT:  The brand name of the heritage leisure segment of hotels is WelcomHotel.<|eot_id|>
CORRECT:  ['WelcomHeritage']

OUTPUT:  This is a nutrition survey.<|eot_id|>
CORRECT:  ['Preliminary']

OUTPUT:  The check amount is $0.00.<|eot_id|>
CORRECT:  ['$ 0.00']

OUTPUT:  The G